# Missing Citations Identifier - MVP Pipeline
This notebook isolates the end-to-end extraction, concept graphing, and candidate retrieval pipeline experiment without polluting the main codebase.

In [1]:
import sys
import os
import logging

# Add the src directory to the Python path
sys.path.append(os.path.abspath('..'))

from src.pipeline.concept_extractor import ConceptExtractor
from src.pipeline.concept_graph import ConceptGraphBuilder
from src.pipeline.candidate_retriever import SimpleCandidateRetriever

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

c:\Users\sampe\OneDrive\Desktop\facultate\licenta\missing-citations-identifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup the Corpus and Pipeline Components
We use a mock corpus of 4 papers, and initialize the extractor, the graphing tool, and BM25 retriever.

In [2]:
logger.info("Initializing pipeline components...")
extractor = ConceptExtractor()
graph_builder = ConceptGraphBuilder()

# MOCK CORPUS: Papers available to be recommended
corpus = [
    {"id": "p1", "title": "Attention Is All You Need", "abstract": "We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely."},
    {"id": "p2", "title": "BERT: Pre-training of Deep Bidirectional Transformers", "abstract": "We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers."},
    {"id": "p3", "title": "Generative Adversarial Nets", "abstract": "We propose a new framework for estimating generative models via an adversarial process."},
    {"id": "p4", "title": "Long Short-Term Memory", "abstract": "Learning to store information over extended time intervals by recurrent backpropagation takes a very long time, mostly because of insufficient, decaying error backflow."}
]

retriever = SimpleCandidateRetriever(corpus_papers=corpus)

INFO: Initializing pipeline components...
INFO: Use pytorch device_name: cuda:0
INFO: Load pretrained SentenceTransformer: all-MiniLM-L6-v2


## 2. Process Target Paper
We provide the abstract of a mock paper. The system will extract the top KeyBERT concepts from it.

In [3]:
# TARGET PAPER (Our document we want to find missing citations for)
target_abstract = "We introduce a novel architecture based on self-attention that replaces recurrent layers in encoder-decoder setups. It parallelizes computation and achieves state-of-the-art results."

print("--- Extraction ---")
concepts = extractor.extract_concepts(target_abstract, top_n=5)
print("Concepts Extracted:")
for c in concepts:
    print(f"  - {c['label']} (score: {c['score']:.4f})")

--- Extraction ---
Concepts Extracted:
  - recurrent layers encoder (score: 0.7319)
  - recurrent layers (score: 0.6189)
  - attention replaces recurrent (score: 0.5928)
  - replaces recurrent layers (score: 0.5794)
  - layers encoder decoder (score: 0.5734)


## 3. Build Graph and Detect Central Entities
We generate an interconnected graph of the concepts found and compute the PageRank to find the most influential terms.

In [4]:
print("\n--- Build Concept Graph ---")
G = graph_builder.build_graph(concepts)
central_concepts = graph_builder.get_central_concepts(G, top_n=5)

print("Top Central Concepts (PageRank):")
for node in central_concepts:
    print(f"  - {node}")


--- Build Concept Graph ---
Top Central Concepts (PageRank):
  - recurrent layers encoder
  - recurrent layers
  - attention replaces recurrent
  - replaces recurrent layers
  - layers encoder decoder


## 4. Gap Detection and Candidate Retrieval
Simulating gap detection by taking the top missing central concepts and performing a BM25 retrieval.

In [5]:
print("\n--- Gap Detection & Retrieval ---")
# For MVP, assume our cited papers failed to cover the most central concepts
uncovered_concepts = central_concepts[:3]
print(f"Assuming these concepts are 'missing' from cited bibliography: {uncovered_concepts}\n")

recommendations = retriever.retrieve(uncovered_concepts, top_k=2)
print("Recommended Citations based on Gap:")
for rec in recommendations:
    print(f"  -> [{rec['id']}] {rec['title']} (BM25 Score: {rec['score']})")


--- Gap Detection & Retrieval ---
Assuming these concepts are 'missing' from cited bibliography: ['recurrent layers encoder', 'recurrent layers', 'attention replaces recurrent']

Recommended Citations based on Gap:
  -> [p4] Long Short-Term Memory (BM25 Score: 2.2754)
  -> [p2] BERT: Pre-training of Deep Bidirectional Transformers (BM25 Score: 0.8742)
